# IV_08 — Caso integrado: pipeline de alertas

## 1. Objetivo

Integrar features, modelo ML, scoring de riesgo y priorización Pareto en un pipeline único para la línea de molienda.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MOD_DIR = Path.cwd()
os.chdir(MOD_DIR)
DATA_DIR = MOD_DIR / "data"
OUTPUT_DIR = MOD_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_PATH = DATA_DIR / "dataset_predictivo.csv"

from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split


## 2. Pipeline completo

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
features = [
    "PUMP101.BEARING_TEMP", "PUMP101.VIBRATION_RMS",
    "PUMP101.DISCHARGE_PRESS", "PUMP101.MOTOR_CURRENT",
    "vib_media_24h", "temp_pendiente_24h",
]
X = df[features]
y_clf = df["falla"]

# Modelo clasificación
clf = RandomForestClassifier(n_estimators=100, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
clf.fit(X_train, y_train)
prob_falla = clf.predict_proba(X)[:, 1]

# RUL y scoring
rul = np.clip(500 - df["PUMP101.VIBRATION_RMS"] * 80 - df["PUMP101.BEARING_TEMP"] * 2, 10, 500)
riesgo = np.clip(prob_falla * 100, 0, 100)

alertas = pd.DataFrame({
    "Timestamp": df["Timestamp"],
    "Equipo": "PUMP101",
    "Prob_Falla": prob_falla,
    "RUL_horas": rul,
    "Riesgo_pct": riesgo,
})
alertas.tail()


## 3. Priorización Pareto

In [ ]:
# Ranking por riesgo (últimas 24 lecturas)
ultimas = alertas.tail(24).sort_values("Riesgo_pct", ascending=False)
ultimas["Riesgo_acum_pct"] = ultimas["Riesgo_pct"].cumsum() / ultimas["Riesgo_pct"].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(len(ultimas)), ultimas["Riesgo_pct"], color="coral")
axes[0].set_xticks(range(len(ultimas)))
axes[0].set_xticklabels(ultimas["Timestamp"].dt.strftime("%m-%d %Hh"), rotation=45, ha="right")
axes[0].set_ylabel("Riesgo %")
axes[0].set_title("Ranking de alertas — PUMP101")

axes[1].plot(ultimas["Riesgo_acum_pct"].values, "ko-")
axes[1].axhline(80, color="red", linestyle="--", label="80% Pareto")
axes[1].set_xlabel("Ranking")
axes[1].set_ylabel("% riesgo acumulado")
axes[1].set_title("Curva Pareto")
axes[1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "dashboard_riesgo.png", dpi=150)


## 4. Exportación

In [ ]:
out_alertas = OUTPUT_DIR / "alertas_priorizadas.csv"
ultimas.to_csv(out_alertas, index=False)
print(f"Alertas exportadas: {out_alertas}")

resumen = pd.DataFrame({
    "Metrica": ["Riesgo_max_pct", "RUL_min_h", "Prob_falla_max"],
    "Valor": [ultimas["Riesgo_pct"].max(), ultimas["RUL_horas"].min(), ultimas["Prob_Falla"].max()],
})
resumen


## 5. Interpretación para mantenimiento

PUMP101 concentra el mayor riesgo en las últimas 24 h de operación simulada. Ejecutar inspección de vibración (espectro) antes de la falla funcional. Integrar alertas en PI AF y Supabase (Módulo B) para trazabilidad.

## 6. Resumen — Módulo C completo

- Pipeline: datos → features → ML → scoring → Pareto → acción.
- IA predictiva complementa, no reemplaza, el criterio del ingeniero.
- Profundización: Labs 06–08 en la raíz del proyecto.

**Curso Minería 5.0 (6h) completado.**